In [ ]:
import gymnasium as gym
import ale_py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import namedtuple
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation, TimeLimit, RecordVideo, RecordEpisodeStatistics
from collections import deque
import os
import cProfile, pstats
import torch.jit as jit
import pathlib

In [ ]:
class ClipRewardRaw(gym.Wrapper):
   def __init__(self, env, lb, ub):
      gym.Wrapper.__init__(self, env)
      self.episode_score = 0.0
      self.lower_bound = lb
      self.upper_bound = ub
      
   def reset(self, **kwargs):
      obs, info = super().reset(**kwargs)
      self.episode_score = 0.0
      return obs, info
   
   def step(self, action):
      obs, reward, terminated, truncated, info = super().step(action)
      self.episode_score += reward
      if reward > 0:
         clipped_reward = min(self.upper_bound, reward)
      elif reward < 0:
         clipped_reward = max(self.lower_bound, reward)
      else:
         clipped_reward = 0.0
      if terminated or truncated:
         info['score'] = self.episode_score
      return obs, clipped_reward, terminated, truncated, info

class AutoFire(gym.Wrapper):
   def __init__(self, env):
      gym.Wrapper.__init__(self, env)

   def reset(self, **kwargs):
      obs, info = super().reset(**kwargs)
      super().step(1)
      return obs, info
   
   def step(self, action):
      lives_before = self.env.unwrapped.ale.lives()
      state, reward, terminated, truncated, info = super().step(action)
      if info['lives'] < lives_before and not terminated and not truncated:
         super().step(1)
      return state, reward, terminated, truncated, info


In [ ]:
def make_env(game):
   def make():
      env = gym.make('ALE/' + game + '-v5', frameskip=1, repeat_action_probability=0.0) 
      env = ClipRewardRaw(env, -1, 1)
      env = AtariPreprocessing(env, noop_max=30, frame_skip=4, terminal_on_life_loss=False, screen_size=84, grayscale_obs=True, scale_obs=False)
      env = FrameStackObservation(env, stack_size=4)
      env = AutoFire(env)
      env = TimeLimit(env, max_episode_steps=18000)
      return env
   return make   

In [ ]:
GAME = "breakout"
envs = AtariVectorEnv(
    # Required parameters
    game=GAME,
    num_envs=8,
    batch_size=0,
    num_threads=0,
    thread_affinity_offset=0,
    max_num_frames_per_episode=108000,
    repeat_action_probability=0.0,
    full_action_space=False,
    continuous=False,
    continuous_action_threshold=0.5,
    # Preprocessing values
    img_height=84,
    img_width=84,
    stack_num=4,
    frameskip=4,
    maxpool=True,
    noop_max=30,
    episodic_life=False,
    life_loss_info=False,
    reward_clipping=False,
    use_fire_reset=True,
)
eval_env = AtariVectorEnv(
# Required parameters
game=GAME,
num_envs=8,
batch_size=0,
num_threads=0,
thread_affinity_offset=0,
max_num_frames_per_episode=108000,
repeat_action_probability=0.0,
full_action_space=False,
continuous=False,
continuous_action_threshold=0.5,
# Preprocessing values
img_height=84,
img_width=84,
stack_num=4,
frameskip=4,
maxpool=True,
noop_max=30,
episodic_life=False,
life_loss_info=False,
reward_clipping=False,
use_fire_reset=True,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_save_path = os.path.join('.\models', f'{GAME}-model.pt')
best_model_save_path = os.path.join('.\models', f'{GAME}-model-BEST.pt')
torch.backends.cudnn.benchmark = True
scaler = torch.amp.GradScaler('cuda')

In [ ]:
REPLAY_MEMORY_SIZE = 1048576
BATCH_SIZE = 32
EPSILON_START = 1.0
EPSILON_END = 0.1
EPSILON_EVALUATION = 0.05
FINAL_EXPLORATION_FRAME = 1000000
DISCOUNT_FACTOR = 0.99
UPDATE_FREQUENCY = 5000
REPLAY_START_SIZE = 55000
IN_DIMENSIONS = 4
LEARNING_RATE = 0.00025
GRADIENT_MOMENTUM = 0.95
SQUARED_GRADIENT_MOMENTRUM = 0.95
MIN_SQUARED_GRADIENT = 0.0001
TRAINING_LENGTH = 10000000
EPOCH_LENGTH = 25000
LOG_INTERVAL = 25000
SAVE_FREQUENCY = 5000000
ENV_NUMBER = 8
TRAINING_PER_STEP = 2
EVALUATION_FREQUENCY = 250000

In [ ]:
class QNetwork(nn.Module):
   def __init__(self, in_dimensions: int, action_num: int):
      super(QNetwork, self).__init__()
      self.in_dimensions = in_dimensions
      self.action_num = action_num
      self.conv1 = nn.Conv2d(in_channels=self.in_dimensions, out_channels=32, kernel_size=8, stride=4)
      self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)
      self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)
      self.fc1 = nn.Linear(in_features=7*7*64, out_features=512)
      self.fc2 = nn.Linear(in_features=512, out_features=self.action_num)

   def forward(self, x):
      x = F.relu(self.conv1(x))
      x = F.relu(self.conv2(x))
      x = F.relu(self.conv3(x))
      # x = x.view(x.size(0), -1)
      x = x.contiguous().view(x.size(0), -1)
      x = F.relu(self.fc1(x))
      x = self.fc2(x)
      return x


In [ ]:
class ReplayBuffer():
   def __init__(self):
      C,H,W = 4, 84, 84
      self.capacity= REPLAY_MEMORY_SIZE
      self.pos     = 0
      self.full    = False

      folder = pathlib.Path('replay_mmap')

      def make(name, dtype, shape):
         path = folder / f'{name}.dat'
         return np.memmap(path, mode='w+', dtype=dtype, shape=shape)

      self._states_np = make('states', np.uint8, (REPLAY_MEMORY_SIZE, C, H, W))
      self._new_states_np = make('new_state', np.uint8, (REPLAY_MEMORY_SIZE, C, H, W))
      self._actions_np = make('actions', np.int64, (REPLAY_MEMORY_SIZE,))
      self._rewards_np = make('rewards', np.float32, (REPLAY_MEMORY_SIZE,))
      self._overs_np = make('overs', np.uint8, (REPLAY_MEMORY_SIZE,))
      
      self.states = torch.as_tensor(self._states_np, device='cpu')
      self.next_states = torch.as_tensor(self._new_states_np, device='cpu')
      self.actions = torch.as_tensor(self._actions_np, device=device)
      self.rewards = torch.as_tensor(self._rewards_np, device=device)
      self.overs = torch.as_tensor(self._overs_np, device=device)

   def append(self, state, action, reward, next_state, over, episode_start):
      for k in range(ENV_NUMBER):
         if not episode_start[k]:
            i = self.pos
            self.states[i].copy_(state[k])
            self.actions[i] = action[k]
            self.rewards[i] = reward[k]
            self.next_states[i].copy_(next_state[k])
            self.overs[i] = over[k]
            self.pos = (i+1)%REPLAY_MEMORY_SIZE
            self.full = self.full or self.pos==0

   def sample(self):
      size = self.capacity if self.full else self.pos
      indexes = torch.randint(size, (BATCH_SIZE,))
      states = self.states.index_select(0, indexes).float().div(255.0).to(device, non_blocking=True, memory_format=torch.channels_last)
      next_states = self.next_states.index_select(0, indexes).float().div(255.0).to(device, non_blocking=True, memory_format=torch.channels_last)
      indexes = indexes.to(device)
      actions = self.actions.index_select(0, indexes)
      rewards = self.rewards.index_select(0, indexes)
      overs   = self.overs.index_select(0, indexes)

      return states, actions, rewards, next_states, overs, self.pos

In [ ]:
class DQNAgent():
   def __init__(self, env, current_state, buffer: 'ReplayBuffer', model: 'QNetwork', target_model: 'QNetwork', optimizer: torch.optim.RMSprop):
      self.env = env
      self.current_state = current_state
      self.replay_memory = buffer
      self.model = model
      self.target_model = target_model
      self.optimizer = optimizer
      self.frames_history   = []    
      self.avg100_history   = []      
      self.reward_window = deque(maxlen=100)
      self.episode_reward = np.zeros(ENV_NUMBER)
      self.episode_start = np.zeros(ENV_NUMBER)
      self.print_cycle = 0
      self.best_average = -100000000000

   def select_action(self, epsilon, state):
      # pick action with epsilon-greedy policy
      if np.random.rand() <= epsilon:
         action = self.env.action_space.sample()
      else:
         self.model.eval()
         state = state.float().div(255.0).to(device)
         with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            with torch.no_grad():
                  actions = self.model(state)
                  action = torch.argmax(actions, dim=1)
                  action = action.cpu().numpy()
            self.model.train()
      return action

   def learn(self):
      states_tensor, actions_tensor, rewards_tensor, next_states_tensor, overs_tensor, pos = self.replay_memory.sample()
      with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
         q_values = self.model(states_tensor)
         q_s_a = q_values.gather(1, actions_tensor.unsqueeze(1))
         # calulcate return
         with torch.no_grad():
            next_q_values = self.target_model(next_states_tensor)
            max_nextvalues = next_q_values.max(dim=1).values.unsqueeze(1)
            y = rewards_tensor.unsqueeze(1) + (1-overs_tensor.unsqueeze(1))*DISCOUNT_FACTOR*max_nextvalues
            y = y.to(q_s_a.dtype) 
         # calculate loss and gradient and move weights according to it
         loss = F.mse_loss(y, q_s_a, reduction='mean')
      self.optimizer.zero_grad()
      scaler.scale(loss).backward()
      scaler.step(self.optimizer)
      scaler.update()
      return loss.item()

   def q_learning(self, state, epsilon, frame):
      # select action and do step in env
      self.print_cycle = (self.print_cycle+1) % 1000
      action = self.select_action(epsilon, state)
      self.env.send(action)

      # sample from buffer and extract values for action pair
      losses = np.zeros(TRAINING_PER_STEP)
      for i in range(TRAINING_PER_STEP):
         losses[i] = self.learn()

      next_state, reward, terminated, truncated, info = self.env.recv()
      self.episode_reward = np.add(self.episode_reward, reward)
      reward = np.clip(reward, -1, 1)
      over = np.empty((ENV_NUMBER))
      for i in range(ENV_NUMBER):
         over[i] = terminated[i] or truncated[i]

      next_state = torch.from_numpy(next_state)
      action = torch.from_numpy(action)
      reward = torch.from_numpy(reward)
      over = torch.from_numpy(over)

      # store the transition in buffer
      self.replay_memory.append(state, action, reward, next_state, over, self.episode_start)

      return losses, next_state, over, info

   def evaluate(self):
      obs, _ = eval_env.reset()
      self.model.eval()
      episode_rewards = np.zeros(ENV_NUMBER)
      episodes = 0
      frame_num = 0
      max_reward = 0
      cum_reward = 0
      q_value_sum = 0
      while frame_num <= 500000:
         frame_num += ENV_NUMBER
         obs = torch.from_numpy(obs).float().div(255.0).to(device)
         if np.random.rand() <= EPSILON_EVALUATION:
            action = eval_env.action_space.sample()
         else:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
               with torch.no_grad():
                  actions = self.model(obs)
                  q_value_sum += torch.sum(torch.max(actions, dim=1)[0]).item()
                  action = torch.argmax(actions, dim=1)
                  action = action.cpu().numpy()
         obs, rew, ter, trun, info = eval_env.step(action)
         episode_rewards = np.add(episode_rewards, rew)
         over = np.empty((ENV_NUMBER))
         for i in range(ENV_NUMBER):
            over[i] = ter[i] or trun[i]
         if over.any():
            for i in range(len(over)):
               if over[i]:
                  episodes += 1
                  cum_reward += episode_rewards[i]
                  if episode_rewards[i] > max_reward:
                     max_reward = episode_rewards[i]
                  episode_rewards[i] = 0
      print(f'Average: {cum_reward/episodes}, Max: {max_reward}')
      print(f'Q Value Estimates: {q_value_sum/500000}')
      if (cum_reward/episodes) > self.best_average:
         self.best_average = cum_reward/episodes
         torch.save(self.model.state_dict(), best_model_save_path)
      self.model.train()

   def train(self):
      frame = 0
      count = 0
      losses = np.zeros(TRAINING_PER_STEP)
      episodes = 0
      epsilon = EPSILON_START
      self.current_state = torch.from_numpy(self.current_state)
      checkpoints = [LOG_INTERVAL*(x+1) for x in range(int(TRAINING_LENGTH/LOG_INTERVAL))]

      while frame < TRAINING_LENGTH:
         frame += ENV_NUMBER
         count += 1

         # fill replay buffer with start samples
         if frame <= REPLAY_START_SIZE:
            action = self.env.action_space.sample()
            next_state, reward, terminated, truncated, info = self.env.step(action)
            self.episode_reward = np.add(self.episode_reward, reward)
            reward = np.clip(reward, -1, 1)
            over = np.empty((ENV_NUMBER))
            for i in range(ENV_NUMBER):
               over[i] = terminated[i] or truncated[i]

            next_state = torch.from_numpy(next_state)
            action = torch.from_numpy(action)
            reward = torch.from_numpy(reward)
            over = torch.tensor(over, dtype=torch.uint8)

            self.replay_memory.append(self.current_state, action, reward, next_state, over, self.episode_start)
            self.current_state = next_state
            self.episode_start = over
         
         else:
            epsilon = max(EPSILON_END, EPSILON_START - (frame-REPLAY_START_SIZE)/FINAL_EXPLORATION_FRAME)
            losses, next_state, over, info = self.q_learning(self.current_state, epsilon, frame)
            self.current_state = next_state
            self.episode_start = over
            
         # reset if game is over
         if over.any():
            for i in range(len(over)):
               if over[i]:
                  self.reward_window.append(self.episode_reward[i])
                  self.episode_reward[i] = 0
            if frame >= checkpoints[0]:
               average_score = np.mean(self.reward_window)
               self.frames_history.append(frame)
               self.avg100_history.append(average_score)
               print(f"frame {frame:>8d} | avg_100_eps {average_score:6.2f} | losses {losses}")
               checkpoints.pop(0)
            episodes += 1

         # periodically update target network weights
         if count % UPDATE_FREQUENCY == 0:
            self.target_model.load_state_dict(self.model.state_dict())

         # if frame % SAVE_FREQUENCY == 0:
         #    torch.save(self.model.state_dict(), model_save_path)

         if frame % EVALUATION_FREQUENCY == 0:
            self.evaluate()
         
      return self.frames_history, self.avg100_history


In [ ]:
action_num = int(envs.single_action_space.n)
model = QNetwork(IN_DIMENSIONS, action_num).to(device)
#sample = torch.randn(1, 4, 84, 84, device="cuda")
#model = torch.jit.trace(model, sample)
target_model = QNetwork(IN_DIMENSIONS, action_num).to(device)
#target_model = torch.jit.trace(target_model, sample)
target_model.load_state_dict(model.state_dict())
optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE, alpha=SQUARED_GRADIENT_MOMENTRUM, momentum=0.0, eps=MIN_SQUARED_GRADIENT, centered=True)
replay_memory = ReplayBuffer()
torch.optim.RMSprop
current_state, _ = envs.reset(seed=0)
agent = DQNAgent(envs, current_state, replay_memory, model, target_model, optimizer)
frames_history, avg100_history = agent.train()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(frames_history, avg100_history, linewidth=2)
plt.xlabel("Frame")
plt.ylabel("Average reward (last 100 episodes)")
plt.title("Training performance")
plt.grid(True)
plt.show()
print("frames logged:", frames_history[:5])
print("avg-100 logged:", avg100_history[:5])

In [ ]:
action_num = int(envs.single_action_space.n)

best_model = QNetwork(IN_DIMENSIONS, action_num).to(device)
best_model.load_state_dict(torch.load(best_model_save_path, weights_only=True))
best_model.eval()

In [ ]:
obs, _ = eval_env.reset()
episode_reward = np.zeros(ENV_NUMBER)
episodes = 0
max_reward = 0
cum_reward = 0
while episodes < 200:
   obs = torch.from_numpy(obs).float().div(255.0).to(device)
   if np.random.rand() <= 0.001:
      action = eval_env.action_space.sample()
   else:
      with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
         with torch.no_grad():
            actions = best_model(obs)
            action = torch.argmax(actions, dim=1)
            action = action.cpu().numpy()
   obs, rew, ter, trun, info = eval_env.step(action)
   episode_reward = np.add(episode_reward, rew)
   over = np.empty((ENV_NUMBER))
   for i in range(ENV_NUMBER):
      over[i] = ter[i] or trun[i]
   if over.any():
      for i in range(len(over)):
         if over[i]:
            episodes += 1
            cum_reward += episode_reward[i]
            if episode_reward[i] > max_reward:
               max_reward = episode_reward[i]
            episode_reward[i] = 0
print(cum_reward/200)
print(max_reward)
      
   

In [ ]:
import cv2
from typing import Optional

class UpscaleRender(gym.Wrapper):
    def __init__(self, env, scale: int = 4):
        super().__init__(env)
        self.scale = int(scale)

    def render(self):                    
        frame = self.env.render()        
        if frame is None:
            return None
        h, w, _ = frame.shape
        return cv2.resize(
            frame,
            (w * self.scale, h * self.scale),
            interpolation=cv2.INTER_NEAREST,
        )

In [ ]:
GAME = "kangaroo"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_save_path = os.path.join('.\models', f'{GAME}-model.pt')
best_model_save_path = os.path.join('.\models', f'{GAME}-model-BEST.pt')
torch.backends.cudnn.benchmark = True
scaler = torch.amp.GradScaler('cuda')

In [ ]:
gym.register_envs(ale_py)
env = gym.make(f'ALE/{GAME.capitalize()}-v5', frameskip=1, repeat_action_probability=0.0, render_mode="rgb_array") 
env = UpscaleRender(env, scale=4)  
env = RecordVideo(
    env,
    video_folder="agent_videos",  
    name_prefix=f"{GAME}",           
    episode_trigger=lambda x: True, 
    fps=60
)
env = AtariPreprocessing(env, noop_max=30, frame_skip=4, terminal_on_life_loss=False, screen_size=84, grayscale_obs=True, scale_obs=False)
env = FrameStackObservation(env, stack_size=4)
env = AutoFire(env)
env = TimeLimit(env, max_episode_steps=18000)

num_eval_episodes = 1
# Add episode statistics tracking
env = RecordEpisodeStatistics(env, buffer_length=num_eval_episodes)

action_num = int(env.action_space.n)

best_model = QNetwork(IN_DIMENSIONS, action_num).to(device)
best_model.load_state_dict(torch.load(best_model_save_path, weights_only=True))
best_model.eval()

obs, _ = env.reset()
episode_reward = 0
episodes = 0
max_reward = 0
cum_reward = 0
while episodes < num_eval_episodes:
   obs = torch.from_numpy(obs).float().div(255.0).to(device)
   obs = obs.unsqueeze(0).to(device)
   if np.random.rand() <= EPSILON_EVALUATION:
      action = env.action_space.sample()
   else:
      with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
         with torch.no_grad():
            actions = best_model(obs)
            action = torch.argmax(actions, dim=1).item()
   obs, rew, ter, trun, info = env.step(action)
   episode_reward += rew
   over = ter or trun
   if over:
        episodes += 1
        print(f"Episode {episodes}: reward = {episode_reward}")
        episode_reward = 0
        obs, _ = env.reset()
        env.close()
        
